# Task A -- using the external corpus's labels

This is the one opening Task A has that Task B did not. `data/external/offenseval_kn.csv`
carries `Hate` / `Non-Hate` labels, which is **exactly** Task A's label space. On Task B
the six-way target taxonomy made those labels unusable and only the text was ever used, in
TAPT. Here the labels can be trained on directly.

Measured today on the TF-IDF floor, five-fold: adding the external rows moves macro-F1
from `0.8073` to `0.8103`. Small, but positive, and transformers gain more from extra data
than linear models do. It adds 3,247 rows to 6,401, a 51% increase.

| arm | flag | what it does |
|---|---|---|
| control | none | Task A rows only, the reference |
| mix | `--external --external-mode mix` | external rows appended to each **training** fold, never to a validation fold |
| stage | `--external --external-mode stage` | fine-tune on the external corpus first, then on Task A from those weights |

Everything else is identical: demojized MuRIL, `--reinit-layers 1`, 6 epochs, effective
batch 16, five folds on split seed 42, `--select last`.

## The rules question, stated plainly

This uses **external labels**, not just external text. TAPT deliberately avoids that, and
every Task B decision so far has kept external labels out. Decide before submitting
anything built on the winning arm. The measurement itself commits you to nothing.

## Why two modes rather than one

The external labels come from a different annotation project, where the positive class is
"offensive targeted insult" rather than "hate". `mix` leaves that label noise in the final
loss. `stage` lets a second pass on clean Task A labels overwrite the boundary the
external corpus sets, which is why it was the default on Task B. Which wins here is an
empirical question, so both run.

## Runtime

About **9.9 hours** for all three: 160 minutes for control, about 240 for mix since it
trains on half again as many rows, and about 195 for stage. The guard runs control and mix
first, which is the pair that answers the question.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Check what the external file actually contains

Printed before training so the arms below are interpretable, and so a corrupted or
swapped file is caught immediately rather than silently changing the result.

In [ ]:
import time
t0 = time.time()
BUDGET_H, RESERVE_MIN = 10.5, 20
left = lambda: BUDGET_H * 3600 - (time.time() - t0) - RESERVE_MIN * 60

from sklearn.metrics import f1_score
from hastika.common.preprocessing import clean
df = train.iloc[keep].reset_index(drop=True)
X = np.array([clean(t, demojize=True) for t in df["Comment"]])
y = (df["Label"] == "Hate").astype(int).values
oof_of = lambda tag: np.load(pathlib.Path("artifacts/runs") / tag / "oof_probs.npy")
score_of = lambda tag: f1_score(y, oof_of(tag).argmax(1), average="macro")

ext = pd.read_csv("data/external/offenseval_kn.csv")
print(f"external rows: {len(ext)}")
print("  label counts :", ext["Label"].value_counts().to_dict())
print("  source labels:", ext["source_label"].value_counts().to_dict())
print(f"\ntask A rows: {len(y)}, {y.mean():.3f} Hate")
print(f"external Hate rate: {(ext['Label'] == 'Hate').mean():.3f}")
print("\nthe external corpus is far more imbalanced than Task A; `mix` therefore")
print("shifts the training prior, which is part of what is being measured")
assert len(ext) == 3247, len(ext)

## 2. Run the three arms

`--external` with no path defaults to `data/external/offenseval_kn.csv`. In `mix` mode the
rows go into training folds only; a validation fold never contains an external row, so the
out-of-fold score stays a measurement of Task A performance.

In [ ]:
COMMON = ["--model", "google/muril-base-cased", "--folds", "5", "--epochs", "6",
          "--bs", "8", "--grad-accum", "2", "--eval-bs", "32",
          "--select", "last", "--reinit-layers", "1", "--seeds", "42"]
ARMS = [("task_a_ext_control", [],                                          160),
        ("task_a_ext_mix",     ["--external", "--external-mode", "mix"],    240),
        ("task_a_ext_stage",   ["--external", "--external-mode", "stage"],  195)]
ran = []
for tag, extra, est in ARMS:
    if left() < est * 60:
        print(f"skip {tag}: {left()/60:.0f} min left, needs ~{est}", flush=True)
        continue
    run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", tag,
         *COMMON, *extra], log=f"artifacts/logs/{tag}.log")
    ran.append(tag)
print("\narms completed:", ran)

## 3. Compare

All figures are out-of-fold macro-F1 over the same 6,401 Task A rows. External rows are
never scored, so the three numbers are directly comparable.

In [ ]:
from sklearn.metrics import classification_report
base = score_of("task_a_ext_control") if "task_a_ext_control" in ran else None
for tag in ran:
    s = score_of(tag)
    d = "" if base is None or tag == "task_a_ext_control" else f"   {s - base:+.4f} vs control"
    print(f"  {tag:22s} OOF macro-F1 {s:.4f}{d}")
print("\n  TF-IDF reference measured today: 0.8073 control, 0.8103 with external rows")
print("  fold noise here is about 0.006")
for tag in ran:
    print(f"\n=== {tag} ===")
    print(classification_report(y, oof_of(tag).argmax(1),
                                target_names=["Non-Hate", "Hate"], digits=3))

## 4. Preserve outputs

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_external_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for tag in ran:
    for name in ["oof_probs.npy", "test_probs.npy", "predictions.csv"]:
        p = pathlib.Path("artifacts/runs") / tag / name
        if p.exists():
            shutil.copy2(p, OUT / f"{tag}_{name}")
    shutil.copy2(f"artifacts/logs/{tag}.log", OUT / f"{tag}.log")
json.dump({t: score_of(t) for t in ran}, open(OUT / "oof_scores.json", "w"), indent=2)
print(sorted(x.name for x in OUT.iterdir()))

## 5. What to do with the result

Record all three out-of-fold numbers in `docs/EXPERIMENTS.md`, including the losers.

If a external arm wins by more than about a point, resolve the rules question before
building a submission on it, and say in the system paper exactly which external labels
were used and how. If both external arms land inside noise, the answer is that Task A has
enough data of its own and the corpus is worth only its text, which is the same conclusion
Task B reached by a different route.

No submission is produced here. This is a measurement.